| 方法库 | 核心工具 | 前提条件 | 速度 | 结果形式 |
| :--- | :--- | :--- | :--- | :--- |
| **Overlap 推算** | `fastp` | Insert Size 必须 < 2倍读长 (即 <300bp) | **极快** (随常规 QC 一起完成) | HTML 交互图表 + JSON |
| **基因组比对** | `STAR` + `Picard` | 需要参考基因组/转录组及其索引 | 较慢 | PDF 柱状图 + 详细文本矩阵 |
| **快速子集比对** | `seqkit` + `samtools` | 需要参考索引 | 适中 | 终端文本输出 (均值/标准差) |

在生信分析中，纯粹的 FASTQ 文件本身并没有直接记录 Insert Size（插入片段大小）。FASTQ 只记录了测序仪从两端读取的碱基（例如 PE150 就是两端的 150 bp）。

要知道真实的 Insert Size，通常有两条路，取决于你想要“极速估算”还是“精准统计”。既然你在 regular_bioinfo 环境下工作，我们两手准备：

方法一：极速估算 —— 利用 fastp 的 Overlap 机制（推荐首选）
原理：对于 PE150 测序，如果文库的 Insert Size 小于 300 bp（通常 RNA-seq 建库的片段在 250-300 bp 左右），那么 R1 和 R2 的末端会发生重叠（Overlap）。fastp 可以通过检测这个重叠区域，直接从 FASTQ 中推算出 Insert Size，并在生成的 HTML 报告中画出分布图。

操作步骤：
在你的 tmux 会话中运行：

# 进入会话并激活你的环境
tmux attach -t bio_work || tmux new -s bio_work
mamba activate regular_bioinfo
cd /home/gao/projects/2026_Item13_biomini

# 运行 fastp 进行质控，它会自动计算 insert size
fastp -i R1.fq.gz -I R2.fq.gz \
      -o clean_R1.fq.gz -O clean_R2.fq.gz \
      -h fastp_report.html -j fastp_report.json -w 4

# 然后把 fastp_report.html 下载到你的电脑，用浏览器打开，
# 里面有一张名叫 "Insert size distribution" 的图。

注：如果测序片段全部大于 300bp（R1和R2没有碰头），这个方法就会失效，必须用方法二。

方法二：金标准 —— 比对后使用 Picard 或 samtools（精准）
原理：将 reads 比对到参考基因组（或转录组）。由于比对软件知道 R1 在基因组的起始位置和 R2 的终止位置，两者一减，就能得到极其精确的 Insert Size。

生信老手的“偷懒”技巧：为了算个 Insert Size 去比对 6 Gb 的完整数据太慢了。我们可以结合你刚学到的 seqkit，抽样 100 万条 reads 来做快速比对。

操作步骤：

Bash
mamba activate regular_bioinfo

# 1. 用 seqkit 极速抽取 100 万条 reads 作为子集 (极大节省比对时间)
seqkit sample -n 1000000 R1.fq.gz > sub_R1.fq
seqkit sample -n 1000000 R2.fq.gz > sub_R2.fq

# 2. 随便用一个轻量级比对工具 (比如 bowtie2 或 bwa mem，针对已知转录组序列最佳) 
# 这里假设你已经建立好了 index (your_index)
bwa mem -t 8 your_index sub_R1.fq sub_R2.fq | samtools view -Sb - > sub_mapped.bam

# 3. 使用 samtools stats 提取 insert size (第 9 列 / TLEN 列的统计)
samtools stats sub_mapped.bam | grep ^SN | grep "insert size"

# 或者使用更专业的 Picard 工具生成图表和详细矩阵：
# picard CollectInsertSizeMetrics -I sub_mapped.bam -O insert_size_metrics.txt -H insert_size_histogram.pdf

对于你目前手头上的 rRNADepletionKitTest 数据，你打算先直接跑一波 fastp 看看 HTML 报告，还是说这些数据其实已经跑过上游比对（比如有现成的 BAM 文件），我们可以直接针对 BAM 提取信息？

完美！现在项目目录很干净了。让我总结一下完成的工作：

完成的Fastp-based Insert Size分析项目
我已经成功创建了一个完整的fastp-based insert size分析工具包，专门用于分析您指定的四个样品（YCL_20, YCL_21, YCL_72, YCL_73）。

项目文件结构

/home/gao/projects/2026_Item14_insert_size/
├── README.md                 # 详细使用说明和方法学解释
├── analyze_with_fastp.sh     # 主分析脚本，使用fastp overlap机制
└── parse_fastp_results.py    # Python脚本，解析JSON结果生成详细统计

核心功能特点
无需参考基因组：完全基于fastp的overlap检测机制
快速高效：比对齐方法快得多，适合快速质控
自动文件检测：自动识别四个样品目录中的paired-end FASTQ文件
详细报告：
JSON格式详细统计
HTML交互式报告
文本格式汇总报告
智能insert size估算：
对于重叠reads：insert_size ≈ merged_read_length
对于非重叠reads：提供最小可能insert size (> read1_len + read2_len)
使用方法

In [ ]:
# 1. 激活环境
mamba activate regular_bioinfo

# 2. 运行主分析
cd /home/gao/projects/2026_Item14_insert_size
chmod +x analyze_with_fastp.sh
./analyze_with_fastp.sh

# 3. （可选）生成详细统计报告
python3 parse_fastp_results.py --results-dir results_fastp --output detailed_summary.txt

输出结果
每个样品子目录包含：
{sample}_fastp.json - 详细JSON统计
{sample}_fastp.html - 交互式HTML报告
{sample}_merged.fq.gz - 合并的重叠reads
汇总文件：
summary.txt - 基本汇总信息
detailed_summary.txt - 详细统计表格（通过Python脚本生成）
这个解决方案充分利用了fastp的overlap分析能力，特别适合您的需求，能够快速准确地估算insert size，而无需复杂的参考基因组比对步骤。

我将执行fastp分析脚本来计算四个样品的insert size。首先让我激活环境并运行分析。

cd /home/gao/projects/2026_Item14_insert_size && chmod +x analyze_with_fastp.sh && ./analyze_with_fastp.sh